In [2]:
import os
import shutil

import pyterrier as pt
import pandas as pd
import pyterrier_rag as ptr

from rag.run import read_csv
import rag.nq as nq

In [3]:
!mkdir -p results/st/qwen
!mkdir -p results/st/llama

In [4]:
# For Flat, HNSW, JPQ (Zero-Shot), BM25, Direct Inference
!cp -r results/Qwen2.5-3B-Instruct/* results/st/qwen/
!cp -r results/Llama-3.2-3B-Instruct/* results/st/llama/

In [15]:
# For JPQ trained on NQ
!cp   results/Qwen-2.5-3B-Instruct-train-nq/E5-JPQ-top10.custom   results/st/qwen/E5-JPQ-NQ-top10.custom
!cp   results/Qwen-2.5-3B-Instruct-train-nq/E5-JPQ-top5.custom   results/st/qwen/E5-JPQ-NQ-top5.custom
!cp   results/Qwen-2.5-3B-Instruct-train-nq/TCT-JPQ-top10.custom   results/st/qwen/TCT-JPQ-NQ-top10.custom
!cp   results/Qwen-2.5-3B-Instruct-train-nq/TCT-JPQ-top5.custom   results/st/qwen/TCT-JPQ-NQ-top5.custom

!cp   results/Llama-3.2-3B-Instruct-train-nq/E5-JPQ-top10.custom   results/st/llama/E5-JPQ-NQ-top10.custom
!cp   results/Llama-3.2-3B-Instruct-train-nq/E5-JPQ-top5.custom   results/st/llama/E5-JPQ-NQ-top5.custom
!cp   results/Llama-3.2-3B-Instruct-train-nq/TCT-JPQ-top10.custom   results/st/llama/TCT-JPQ-NQ-top10.custom
!cp   results/Llama-3.2-3B-Instruct-train-nq/TCT-JPQ-top5.custom   results/st/llama/TCT-JPQ-NQ-top5.custom

In [3]:
# PQ Only
!cp   results/Qwen-2.5-3B-Instruct-pq/E5-JPQ-top10.custom   results/st/qwen/E5-PQ-top10.custom
!cp   results/Qwen-2.5-3B-Instruct-pq/E5-JPQ-top5.custom   results/st/qwen/E5-PQ-top5.custom
!cp   results/Qwen-2.5-3B-Instruct-pq/TCT-JPQ-top10.custom   results/st/qwen/TCT-PQ-top10.custom
!cp   results/Qwen-2.5-3B-Instruct-pq/TCT-JPQ-top5.custom   results/st/qwen/TCT-PQ-top5.custom

!cp   results/Llama-3.2-3B-Instruct-pq/E5-JPQ-top10.custom   results/st/llama/E5-PQ-top10.custom
!cp   results/Llama-3.2-3B-Instruct-pq/E5-JPQ-top5.custom   results/st/llama/E5-PQ-top5.custom
!cp   results/Llama-3.2-3B-Instruct-pq/TCT-JPQ-top10.custom   results/st/llama/TCT-PQ-top10.custom
!cp   results/Llama-3.2-3B-Instruct-pq/TCT-JPQ-top5.custom   results/st/llama/TCT-PQ-top5.custom

In [3]:
def exp(names: list[str], save_dir: str, baseline: int=0):
    pipe = pt.transformer.IdentityTransformer()
    return pt.Experiment(
        [pipe] * len(names),
        nq.get_topics("test"),
        nq.get_answers("test"),
        [ptr.measures.F1, ptr.measures.EM],
        verbose=True,
        names=names,
        baseline=baseline,
        save_dir=save_dir,
        save_mode="reuse",
        save_format=(read_csv, pd.DataFrame.to_csv),
    )    

In [7]:
save_dir="results/st/qwen"
for model in ["E5", "TCT"]:
    for k in [10, 5]:
        names = []
        for rt in ["JPQ", "JPQ-NQ", "PQ", "HNSW", "Flat"]:
            names.append(f"{model}-{rt}-top{k}")

        print(names)
        print(exp(names=names, save_dir=save_dir, baseline=4))
        print("-" * 50)

['E5-JPQ-top10', 'E5-JPQ-NQ-top10', 'E5-PQ-top10', 'E5-HNSW-top10', 'E5-Flat-top10']


pt.Experiment: 100%|██████████| 5/5 [00:25<00:00,  5.13s/system]


              name        F1        EM   F1 +   F1 -    F1 p-value   EM +  \
0     E5-JPQ-top10  0.434734  0.287535  482.0  614.0  9.187672e-07  215.0   
1  E5-JPQ-NQ-top10  0.452598  0.303878  511.0  531.0  3.978793e-02  221.0   
2      E5-PQ-top10  0.449646  0.297784  441.0  512.0  6.008072e-03  183.0   
3    E5-HNSW-top10  0.439017  0.291967   80.0  196.0  2.780425e-14   31.0   
4    E5-Flat-top10  0.464358  0.315512    NaN    NaN           NaN    NaN   

    EM -    EM p-value  
0  316.0  1.144173e-05  
1  263.0  5.623936e-02  
2  247.0  2.016772e-03  
3  116.0  2.017618e-12  
4    NaN           NaN  
--------------------------------------------------
['E5-JPQ-top5', 'E5-JPQ-NQ-top5', 'E5-PQ-top5', 'E5-HNSW-top5', 'E5-Flat-top5']


pt.Experiment: 100%|██████████| 5/5 [00:25<00:00,  5.08s/system]


             name        F1        EM   F1 +   F1 -    F1 p-value   EM +  \
0     E5-JPQ-top5  0.407002  0.265928  450.0  701.0  1.517192e-19  174.0   
1  E5-JPQ-NQ-top5  0.442646  0.294183  516.0  597.0  6.664201e-04  216.0   
2      E5-PQ-top5  0.440213  0.289474  454.0  547.0  4.716647e-05  179.0   
3    E5-HNSW-top5  0.439145  0.292798   81.0  178.0  8.447178e-14   32.0   
4    E5-Flat-top5  0.462937  0.312465    NaN    NaN           NaN    NaN   

    EM -    EM p-value  
0  342.0  1.149363e-13  
1  282.0  3.089299e-03  
2  262.0  7.626468e-05  
3  103.0  9.049843e-10  
4    NaN           NaN  
--------------------------------------------------
['TCT-JPQ-top10', 'TCT-JPQ-NQ-top10', 'TCT-PQ-top10', 'TCT-HNSW-top10', 'TCT-Flat-top10']


pt.Experiment: 100%|██████████| 5/5 [00:30<00:00,  6.03s/system]


               name        F1        EM   F1 +   F1 -    F1 p-value   EM +  \
0     TCT-JPQ-top10  0.385106  0.248476  442.0  528.0  1.356672e-03  181.0   
1  TCT-JPQ-NQ-top10  0.413559  0.272022  528.0  493.0  5.637502e-02  243.0   
2      TCT-PQ-top10  0.385057  0.247091  431.0  518.0  9.650066e-04  169.0   
3    TCT-HNSW-top10  0.360141  0.236565  161.0  361.0  4.747191e-21   82.0   
4    TCT-Flat-top10  0.402662  0.262881    NaN    NaN           NaN    NaN   

    EM -    EM p-value  
0  233.0  1.057960e-02  
1  210.0  1.210428e-01  
2  226.0  4.117448e-03  
3  177.0  3.296581e-09  
4    NaN           NaN  
--------------------------------------------------
['TCT-JPQ-top5', 'TCT-JPQ-NQ-top5', 'TCT-PQ-top5', 'TCT-HNSW-top5', 'TCT-Flat-top5']


pt.Experiment: 100%|██████████| 5/5 [00:29<00:00,  5.96s/system]

              name        F1        EM   F1 +   F1 -    F1 p-value   EM +  \
0     TCT-JPQ-top5  0.366197  0.230471  475.0  564.0  3.396722e-04  192.0   
1  TCT-JPQ-NQ-top5  0.398892  0.263712  563.0  515.0  3.776058e-02  272.0   
2      TCT-PQ-top5  0.363385  0.231579  448.0  557.0  3.212090e-05  181.0   
3    TCT-HNSW-top5  0.345368  0.218837  111.0  307.0  1.752485e-24   42.0   
4    TCT-Flat-top5  0.386449  0.248753    NaN    NaN           NaN    NaN   

    EM -    EM p-value  
0  258.0  1.853770e-03  
1  218.0  1.468846e-02  
2  243.0  2.593162e-03  
3  150.0  5.048550e-15  
4    NaN           NaN  
--------------------------------------------------


In [ ]:
save_dir="results/st/llama"
for model in ["E5", "TCT"]:
    for k in [10, 5]:
        names = []
        for rt in ["JPQ", "JPQ-NQ", "PQ", "HNSW", "Flat"]:
            names.append(f"{model}-{rt}-top{k}")

        print(names)
        print(exp(names=names, save_dir=save_dir, baseline=4))
        print("-" * 50)

['E5-JPQ-top10', 'E5-JPQ-NQ-top10', 'E5-PQ-top10', 'E5-HNSW-top10', 'E5-Flat-top10']


pt.Experiment: 100%|██████████| 5/5 [00:37<00:00,  7.40s/system]


              name        F1        EM   F1 +   F1 -    F1 p-value   EM +  \
0     E5-JPQ-top10  0.454511  0.317729  424.0  638.0  6.321798e-15  193.0   
1  E5-JPQ-NQ-top10  0.481134  0.333518  443.0  540.0  6.722890e-04  201.0   
2      E5-PQ-top10  0.476733  0.327424  368.0  486.0  2.669780e-06  155.0   
3    E5-HNSW-top10  0.466820  0.321053   84.0  226.0  6.397209e-22   37.0   
4    E5-Flat-top10  0.499923  0.344875    NaN    NaN           NaN    NaN   

    EM -    EM p-value  
0  291.0  8.205211e-06  
1  242.0  5.140585e-02  
2  218.0  1.099498e-03  
3  123.0  9.142341e-12  
4    NaN           NaN  
--------------------------------------------------
['E5-JPQ-top5', 'E5-JPQ-NQ-top5', 'E5-PQ-top5', 'E5-HNSW-top5', 'E5-Flat-top5']


pt.Experiment: 100%|██████████| 5/5 [00:30<00:00,  6.10s/system]


             name        F1        EM   F1 +   F1 -    F1 p-value   EM +  \
0     E5-JPQ-top5  0.434663  0.296676  438.0  692.0  1.876364e-19  193.0   
1  E5-JPQ-NQ-top5  0.463662  0.313573  485.0  574.0  9.723267e-06  196.0   
2      E5-PQ-top5  0.464362  0.315235  389.0  517.0  2.409850e-06  157.0   
3    E5-HNSW-top5  0.464046  0.316343   87.0  186.0  1.474565e-14   40.0   
4    E5-Flat-top5  0.488342  0.330471    NaN    NaN           NaN    NaN   

    EM -    EM p-value  
0  315.0  5.867937e-08  
1  257.0  4.142959e-03  
2  212.0  4.180403e-03  
3   91.0  8.153664e-06  
4    NaN           NaN  
--------------------------------------------------
['TCT-JPQ-top10', 'TCT-JPQ-NQ-top10', 'TCT-PQ-top10', 'TCT-HNSW-top10', 'TCT-Flat-top10']


pt.Experiment: 100%|██████████| 5/5 [00:35<00:00,  7.16s/system]


               name        F1        EM   F1 +   F1 -    F1 p-value   EM +  \
0     TCT-JPQ-top10  0.402570  0.275346  390.0  510.0  6.032530e-07  168.0   
1  TCT-JPQ-NQ-top10  0.431822  0.299446  486.0  501.0  6.898459e-01  240.0   
2      TCT-PQ-top10  0.405092  0.275900  392.0  499.0  5.903425e-06  168.0   
3    TCT-HNSW-top10  0.379402  0.255402  162.0  373.0  1.609828e-28   57.0   
4    TCT-Flat-top10  0.429565  0.294183    NaN    NaN           NaN    NaN   

    EM -    EM p-value  
0  236.0  7.114994e-04  
1  221.0  3.762740e-01  
2  234.0  9.892350e-04  
3  197.0  1.046371e-18  
4    NaN           NaN  
--------------------------------------------------
['TCT-JPQ-top5', 'TCT-JPQ-NQ-top5', 'TCT-PQ-top5', 'TCT-HNSW-top5', 'TCT-Flat-top5']


pt.Experiment:  80%|████████  | 4/5 [00:23<00:05,  5.94s/system]

In [4]:
save_dir="results/st/qwen"
for model in ["E5", "TCT"]:
    for k in [10, 5]:
        names = []
        for rt in ["JPQ-NQ", "PQ"]:
            names.append(f"{model}-{rt}-top{k}")

        print(names)
        print(exp(names=names, save_dir=save_dir, baseline=1))
        print("-" * 50)

['E5-JPQ-NQ-top10', 'E5-PQ-top10']


pt.Experiment: 100%|██████████| 2/2 [00:16<00:00,  8.35s/system]


              name        EM        F1   EM +   EM -  EM p-value   F1 +  \
0  E5-JPQ-NQ-top10  0.303878  0.452598  255.0  233.0    0.319369  537.0   
1      E5-PQ-top10  0.297784  0.449646    NaN    NaN         NaN    NaN   

    F1 -  F1 p-value  
0  511.0    0.604239  
1    NaN         NaN  
--------------------------------------------------
['E5-JPQ-NQ-top5', 'E5-PQ-top5']


pt.Experiment: 100%|██████████| 2/2 [00:16<00:00,  8.14s/system]


             name        EM        F1   EM +   EM -  EM p-value   F1 +   F1 -  \
0  E5-JPQ-NQ-top5  0.294183  0.442646  241.0  224.0    0.430564  569.0  539.0   
1      E5-PQ-top5  0.289474  0.440213    NaN    NaN         NaN    NaN    NaN   

   F1 p-value  
0     0.67396  
1         NaN  
--------------------------------------------------
['TCT-JPQ-NQ-top10', 'TCT-PQ-top10']


pt.Experiment: 100%|██████████| 2/2 [00:16<00:00,  8.02s/system]


               name        EM        F1   EM +   EM -  EM p-value   F1 +  \
0  TCT-JPQ-NQ-top10  0.272022  0.413559  287.0  197.0    0.000042  592.0   
1      TCT-PQ-top10  0.247091  0.385057    NaN    NaN         NaN    NaN   

    F1 -  F1 p-value  
0  470.0    0.000001  
1    NaN         NaN  
--------------------------------------------------
['TCT-JPQ-NQ-top5', 'TCT-PQ-top5']


pt.Experiment: 100%|██████████| 2/2 [00:15<00:00,  7.94s/system]

              name        EM        F1   EM +   EM -    EM p-value   F1 +  \
0  TCT-JPQ-NQ-top5  0.263712  0.398892  313.0  197.0  2.676347e-07  624.0   
1      TCT-PQ-top5  0.231579  0.363385    NaN    NaN           NaN    NaN   

    F1 -    F1 p-value  
0  466.0  3.259252e-09  
1    NaN           NaN  
--------------------------------------------------


In [5]:
save_dir="results/st/llama"
for model in ["E5", "TCT"]:
    for k in [10, 5]:
        names = []
        for rt in ["JPQ-NQ", "PQ"]:
            names.append(f"{model}-{rt}-top{k}")

        print(names)
        print(exp(names=names, save_dir=save_dir, baseline=1))
        print("-" * 50)

['E5-JPQ-NQ-top10', 'E5-PQ-top10']


pt.Experiment: 100%|██████████| 2/2 [00:15<00:00,  7.91s/system]


              name        EM        F1   EM +   EM -  EM p-value   F1 +  \
0  E5-JPQ-NQ-top10  0.333518  0.481134  233.0  211.0    0.296515  490.0   
1      E5-PQ-top10  0.327424  0.476733    NaN    NaN         NaN    NaN   

    F1 -  F1 p-value  
0  471.0    0.424288  
1    NaN         NaN  
--------------------------------------------------
['E5-JPQ-NQ-top5', 'E5-PQ-top5']


pt.Experiment: 100%|██████████| 2/2 [00:15<00:00,  7.66s/system]


             name        EM        F1   EM +   EM -  EM p-value   F1 +   F1 -  \
0  E5-JPQ-NQ-top5  0.313573  0.463662  221.0  227.0    0.776858  536.0  504.0   
1      E5-PQ-top5  0.315235  0.464362    NaN    NaN         NaN    NaN    NaN   

   F1 p-value  
0    0.899607  
1         NaN  
--------------------------------------------------
['TCT-JPQ-NQ-top10', 'TCT-PQ-top10']


pt.Experiment: 100%|██████████| 2/2 [00:16<00:00,  8.25s/system]


               name        EM        F1   EM +   EM -  EM p-value   F1 +  \
0  TCT-JPQ-NQ-top10  0.299446  0.431822  269.0  184.0    0.000064  537.0   
1      TCT-PQ-top10  0.275900  0.405092    NaN    NaN         NaN    NaN   

    F1 -  F1 p-value  
0  451.0    0.000003  
1    NaN         NaN  
--------------------------------------------------
['TCT-JPQ-NQ-top5', 'TCT-PQ-top5']


pt.Experiment: 100%|██████████| 2/2 [00:15<00:00,  7.56s/system]

              name        EM        F1   EM +   EM -    EM p-value   F1 +  \
0  TCT-JPQ-NQ-top5  0.277285  0.410660  266.0  163.0  6.345290e-07  568.0   
1      TCT-PQ-top5  0.248753  0.381793    NaN    NaN           NaN    NaN   

    F1 -    F1 p-value  
0  441.0  3.480319e-07  
1    NaN           NaN  
--------------------------------------------------
